# Phase 4: Model Training & Evaluation

## Overview
This notebook trains machine learning models to detect NTFS timestomping based on file-level features from Phase 3.

## Models
1. Random Forest
2. XGBoost
3. LightGBM
4. Logistic Regression

## Workflow
1. Load training features from Phase 3
2. Prepare labels (from detection flags)
3. Handle class imbalance (SMOTE, class weights)
4. Train-test split (80/20 stratified)
5. Train all 4 models
6. Evaluate on test set
7. Validate on held-out validation datasets (5 datasets)

## Metrics
- Accuracy, Precision, Recall, F1-Score
- Confusion Matrix (TP, FP, TN, FN)
- ROC-AUC Score
- Classification Report

## Input
- `file_features_training.csv` - Training features (22 datasets)
- `detection_flags_training.csv` - Training labels

## Validation Datasets
- 02-APT19, 09-APT40, 12-Kimsuky, 13-Winnti731, LoneWolf


In [27]:
# [Cell 2] Imports and Configuration

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Joblib for model saving
import joblib
import json

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================

PHASE3_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling")
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training")
DATA_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ground truth file
GROUND_TRUTH_PATH = DATA_DIR / "Suspicious Files.csv"

# Training datasets (22 total)
TRAINING_DATASETS = [
    "01-PE", "02-PE", "03-PE", "04-PE", "05-PE", "06-PE",
    "07-PE", "08-PE", "09-PE", "10-PE", "11-PE", "12-PE",
    "01-APT17", "03-APT21", "04-APT28", "05-APT29", "06-APT30",
    "07-APT37", "08-APT38", "10-DarkHotel663", "11-DarkHotelbbd", "14-Winnti53b"
]

# Validation datasets (5 total)
VALIDATION_DATASETS = ["02-APT19", "09-APT40", "12-Kimsuky", "13-Winnti731", "LoneWolf"]

print("Libraries imported successfully.")
print(f"\nPhase 3 input: {PHASE3_DIR}")
print(f"Ground truth: {GROUND_TRUTH_PATH}")
print(f"Phase 4 output: {OUTPUT_DIR}")
print(f"\nTraining datasets: {len(TRAINING_DATASETS)}")
print(f"Validation datasets: {len(VALIDATION_DATASETS)}")


Libraries imported successfully.

Phase 3 input: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling
Ground truth: /Users/soni/Github/Digital-Detectives_Thesis/data/Suspicious Files.csv
Phase 4 output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training

Training datasets: 22
Validation datasets: 5


## Step 1: Load Ground Truth Labels

Load the actual known timestomped files from `Suspicious Files.csv`.
This contains 44 files identified by Oh et al. using LogTracker tool.


In [28]:
# [Cell 4] Load Ground Truth

print("Loading ground truth labels...")

df_ground_truth = pd.read_csv(GROUND_TRUTH_PATH)
print(f"Ground truth loaded: {len(df_ground_truth)} known timestomped files")

print(f"\nGround truth structure:")
print(df_ground_truth.head(10))

print(f"\nTimestomped files per dataset:")
gt_per_dataset = df_ground_truth.groupby('dataID').size().reset_index(name='count')
print(gt_per_dataset.to_string(index=False))

# Check which datasets have ground truth
training_with_gt = [d for d in TRAINING_DATASETS if d in df_ground_truth['dataID'].values]
validation_with_gt = [d for d in VALIDATION_DATASETS if d in df_ground_truth['dataID'].values]

print(f"\nTraining datasets with ground truth: {len(training_with_gt)}/{len(TRAINING_DATASETS)}")
print(f"  {training_with_gt}")
print(f"\nValidation datasets with ground truth: {len(validation_with_gt)}/{len(VALIDATION_DATASETS)}")
print(f"  {validation_with_gt}")


Loading ground truth labels...
Ground truth loaded: 44 known timestomped files

Ground truth structure:
  dataID                               FileName  is_timestomped
0  01-PE      NewFileTime_SI_C_Manipulation.dll               1
1  02-PE      NewFileTime_SI_M_Manipulation.dll               1
2  03-PE    NewFileTime_SI_MAC_Manipulation.dll               1
3  04-PE       PowerShell_SI_C_Manipulation.dll               1
4  05-PE       PowerShell_SI_M_Manipulation.dll               1
5  06-PE     PowerShell_SI_MAC_Manipulation.dll               1
6  07-PE       nTimestomp_SI_C_Manipulation.dll               1
7  08-PE       nTimestomp_SI_M_Manipulation.dll               1
8  09-PE    nTimestomp_SI_MACE_Manipulation.dll               1
9  11-PE  SetMACE_SI_MACE_Copy_Manipulation.dll               1

Timestomped files per dataset:
         dataID  count
       01-APT17      1
          01-PE      1
          02-PE      1
       03-APT21      1
          03-PE      1
       04-APT28      1

## Step 2: Load and Label Training Data

Load file features from Phase 3 and join with ground truth labels.
Handle the LoneWolf path format specially (ground truth uses `/Dropbox/filename.ext`).

In [29]:
# [Cell 6] Helper Function: Match Ground Truth

def match_ground_truth(df_features, df_gt, dataset_id):
    """
    Match ground truth labels to file features.
    
    For most datasets: match on FileName
    For LoneWolf: match on FilePath (ground truth uses /Dropbox/file.ext format)
    
    Returns DataFrame with is_timestomped column added.
    """
    df = df_features.copy()
    df['is_timestomped'] = 0  # Default: not timestomped
    
    # Get ground truth for this dataset
    gt_dataset = df_gt[df_gt['dataID'] == dataset_id]
    
    if len(gt_dataset) == 0:
        return df, 0  # No ground truth for this dataset
    
    matched_count = 0
    
    if dataset_id == "LoneWolf":
        # LoneWolf: ground truth uses path format like /Dropbox/DeathToll.jpg
        # Match against FilePath column
        for _, gt_row in gt_dataset.iterrows():
            gt_path = gt_row['FileName']  # Actually contains path like /Dropbox/file.jpg
            
            # Match files where FilePath ends with the ground truth path
            # or FilePath contains /Dropbox/ and filename matches
            if gt_path.startswith('/Dropbox/'):
                gt_filename = gt_path.split('/')[-1]  # Extract just the filename
                mask = (df['FilePath'].str.contains('/Dropbox/', na=False)) & \
                       (df['FileName'] == gt_filename)
            else:
                mask = df['FilePath'] == gt_path
            
            matches = mask.sum()
            if matches > 0:
                df.loc[mask, 'is_timestomped'] = 1
                matched_count += matches
    else:
        # Other datasets: match on FileName directly
        for _, gt_row in gt_dataset.iterrows():
            gt_filename = gt_row['FileName']
            mask = df['FileName'] == gt_filename
            matches = mask.sum()
            if matches > 0:
                df.loc[mask, 'is_timestomped'] = 1
                matched_count += matches
    
    return df, matched_count


print("Ground truth matching function defined.")


Ground truth matching function defined.


In [30]:
# [Cell 7] Load and Label All Training Data

print("Loading and labeling training data...")
print("=" * 70)

all_training_data = []
training_label_summary = []

for dataset_id in TRAINING_DATASETS:
    features_path = PHASE3_DIR / f"file_features_{dataset_id}.csv"
    
    if not features_path.exists():
        print(f"  {dataset_id}: SKIPPED (file not found)")
        continue
    
    # Load features
    df_features = pd.read_csv(features_path, low_memory=False)
    
    # Match ground truth
    df_labeled, matched = match_ground_truth(df_features, df_ground_truth, dataset_id)
    
    # Get expected count from ground truth
    expected = len(df_ground_truth[df_ground_truth['dataID'] == dataset_id])
    
    print(f"  {dataset_id}: {len(df_labeled):,} files, {matched} timestomped (expected: {expected})")
    
    all_training_data.append(df_labeled)
    training_label_summary.append({
        'dataID': dataset_id,
        'total_files': len(df_labeled),
        'timestomped_found': matched,
        'timestomped_expected': expected
    })

# Combine all training data
df_train_all = pd.concat(all_training_data, ignore_index=True)

print("=" * 70)
print(f"\nTotal training files: {len(df_train_all):,}")
print(f"Total timestomped files found: {df_train_all['is_timestomped'].sum()}")

# Summary table
df_summary = pd.DataFrame(training_label_summary)
print(f"\nLabel matching summary:")
print(df_summary.to_string(index=False))


Loading and labeling training data...
  01-PE: 37,375 files, 1 timestomped (expected: 1)
  02-PE: 105,251 files, 1 timestomped (expected: 1)
  03-PE: 109,985 files, 1 timestomped (expected: 1)
  04-PE: 7,882 files, 1 timestomped (expected: 1)
  05-PE: 9,451 files, 1 timestomped (expected: 1)
  06-PE: 9,434 files, 1 timestomped (expected: 1)
  07-PE: 110,336 files, 1 timestomped (expected: 1)
  08-PE: 110,541 files, 1 timestomped (expected: 1)
  09-PE: 111,889 files, 1 timestomped (expected: 1)
  10-PE: 110,574 files, 0 timestomped (expected: 0)
  11-PE: 9,415 files, 1 timestomped (expected: 1)
  12-PE: 9,160 files, 1 timestomped (expected: 1)
  01-APT17: 31,455 files, 1 timestomped (expected: 1)
  03-APT21: 28,028 files, 1 timestomped (expected: 1)
  04-APT28: 29,399 files, 1 timestomped (expected: 1)
  05-APT29: 29,860 files, 6 timestomped (expected: 6)
  06-APT30: 22,266 files, 1 timestomped (expected: 1)
  07-APT37: 29,426 files, 1 timestomped (expected: 1)
  08-APT38: 20,478 files,

In [31]:
# [Cell 8] Analyze Class Imbalance

print("=" * 70)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 70)

y_all = df_train_all['is_timestomped']

n_negative = (y_all == 0).sum()
n_positive = (y_all == 1).sum()

print(f"\nClass distribution:")
print(f"  Class 0 (Normal):     {n_negative:,} ({n_negative / len(y_all) * 100:.4f}%)")
print(f"  Class 1 (Timestomped): {n_positive:,} ({n_positive / len(y_all) * 100:.6f}%)")

if n_positive > 0:
    imbalance_ratio = n_negative / n_positive
    print(f"\nImbalance ratio: {imbalance_ratio:,.0f}:1")
    print(f"\nThis is EXTREME imbalance - {imbalance_ratio:,.0f}x more normal files than timestomped!")
else:
    imbalance_ratio = float('inf')
    print("\nWARNING: No timestomped files found in training data!")


CLASS IMBALANCE ANALYSIS

Class distribution:
  Class 0 (Normal):     970,040 (99.9970%)
  Class 1 (Timestomped): 29 (0.002989%)

Imbalance ratio: 33,450:1

This is EXTREME imbalance - 33,450x more normal files than timestomped!


## Step 3: Prepare Features and Labels

Select feature columns and prepare the feature matrix.
Exclude identifier columns and any computed flag columns.


In [32]:
# [Cell 10] Select Features

# Define columns to exclude (identifiers and computed flags)
EXCLUDE_COLUMNS = [
    'FileFRN', 'dataID', 'FileName', 'FilePath',
    'is_timestomped',  # This is our target
    # Exclude any flag columns if they exist
    'flag_backward_timestamp', 'flag_creation_changed', 'flag_zero_nanoseconds',
    'flag_logfile_usn_mismatch', 'flag_usn_basic_pattern', 'flag_repeated_si_update',
    'flag_only_si_modified', 'flag_potential_timestomp', 'flag_silent_timestomp',
    'flag_high_suspicion', 'suspicion_score'
]

# Get feature columns
feature_columns = [col for col in df_train_all.columns if col not in EXCLUDE_COLUMNS]

# Create feature matrix
X = df_train_all[feature_columns].copy()

# Handle NaN values
X = X.fillna(0)

# Convert boolean columns to int
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

# Target variable
y = df_train_all['is_timestomped'].astype(int)

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {len(feature_columns)}")
print(f"\nFeature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {col}")


Feature matrix shape: (970069, 29)
Number of features: 29

Feature columns:
   1. num_timestamp_changes
   2. num_backward_jumps
   3. num_forward_jumps
   4. num_creation_changes
   5. max_backward_jump_seconds
   6. mean_jump_seconds
   7. timestamp_change_density
   8. num_zero_nanosecond_events
   9. only_SI_modified
  10. num_update_resident_value
  11. repeated_update_resident_value
  12. consecutive_timestamp_changes
  13. num_logfile_events
  14. has_logfile_ts_change
  15. has_usn_basic_info
  16. has_usn_close
  17. has_usn_file_create
  18. num_usn_basic_info
  19. num_usn_close
  20. num_usn_file_create
  21. logfile_usn_mismatch
  22. has_usn_basic_pattern
  23. num_usnjrnl_events
  24. min_inter_event_delta
  25. max_inter_event_delta
  26. mean_inter_event_delta
  27. burstiness_score
  28. event_time_span_seconds
  29. total_events


## Step 4: Train-Test Split

Split data into training (80%) and test (20%) sets using stratified sampling.
With extreme imbalance, stratification ensures both sets have representative samples.


In [33]:
# [Cell 12] Train-Test Split

# Check if we have enough positive samples for stratification
if y.sum() < 2:
    print("ERROR: Not enough positive samples for train-test split!")
    print("Need at least 2 positive samples.")
else:
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    print("Train-Test Split Complete")
    print(f"\nTraining set: {len(X_train):,} samples")
    print(f"  Class 0 (Normal):     {(y_train == 0).sum():,} ({(y_train == 0).mean() * 100:.4f}%)")
    print(f"  Class 1 (Timestomped): {(y_train == 1).sum():,} ({(y_train == 1).mean() * 100:.6f}%)")
    
    print(f"\nTest set: {len(X_test):,} samples")
    print(f"  Class 0 (Normal):     {(y_test == 0).sum():,} ({(y_test == 0).mean() * 100:.4f}%)")
    print(f"  Class 1 (Timestomped): {(y_test == 1).sum():,} ({(y_test == 1).mean() * 100:.6f}%)")


Train-Test Split Complete

Training set: 776,055 samples
  Class 0 (Normal):     776,032 (99.9970%)
  Class 1 (Timestomped): 23 (0.002964%)

Test set: 194,014 samples
  Class 0 (Normal):     194,008 (99.9969%)
  Class 1 (Timestomped): 6 (0.003093%)


In [34]:
# [Cell 13] Handle Class Imbalance - Multiple Strategies

print("=" * 70)
print("HANDLING EXTREME CLASS IMBALANCE")
print("=" * 70)

# Strategy 1: SMOTE with controlled ratio
# Don't oversample to 1:1 - that creates too many synthetic samples
# Instead, aim for a more reasonable ratio like 1:10 or 1:100

n_positive_train = (y_train == 1).sum()
n_negative_train = (y_train == 0).sum()

print(f"\nOriginal training distribution:")
print(f"  Positive: {n_positive_train}")
print(f"  Negative: {n_negative_train}")
print(f"  Ratio: {n_negative_train / n_positive_train:.0f}:1")

# Calculate target for SMOTE - aim for 1:100 ratio
target_positive = min(n_negative_train // 100, n_positive_train * 100)
target_positive = max(target_positive, n_positive_train)  # At least keep original

print(f"\nApplying SMOTE...")
print(f"  Target positive samples: {target_positive}")

if n_positive_train > 0:
    smote = SMOTE(
        sampling_strategy={1: target_positive},
        random_state=42,
        k_neighbors=min(5, n_positive_train - 1) if n_positive_train > 1 else 1
    )
    
    try:
        X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
        print(f"\nAfter SMOTE:")
        print(f"  Positive: {(y_train_balanced == 1).sum()}")
        print(f"  Negative: {(y_train_balanced == 0).sum()}")
        print(f"  New ratio: {(y_train_balanced == 0).sum() / (y_train_balanced == 1).sum():.0f}:1")
    except Exception as e:
        print(f"  SMOTE failed: {e}")
        print("  Using original data with class weights instead.")
        X_train_balanced = X_train.copy()
        y_train_balanced = y_train.copy()
else:
    print("  No positive samples - skipping SMOTE")
    X_train_balanced = X_train.copy()
    y_train_balanced = y_train.copy()

# Strategy 2: Calculate class weights for algorithms that support it
if n_positive_train > 0:
    # Higher weight for minority class to prioritize RECALL
    # Using sqrt of imbalance ratio to avoid extreme weights
    weight_ratio = np.sqrt(n_negative_train / n_positive_train)
    class_weight_dict = {0: 1.0, 1: weight_ratio}
    scale_pos_weight = weight_ratio
else:
    class_weight_dict = {0: 1.0, 1: 1.0}
    scale_pos_weight = 1.0

print(f"\nClass weights for model training:")
print(f"  Class 0: {class_weight_dict[0]:.2f}")
print(f"  Class 1: {class_weight_dict[1]:.2f}")


HANDLING EXTREME CLASS IMBALANCE

Original training distribution:
  Positive: 23
  Negative: 776032
  Ratio: 33741:1

Applying SMOTE...
  Target positive samples: 2300

After SMOTE:
  Positive: 2300
  Negative: 776032
  New ratio: 337:1

Class weights for model training:
  Class 0: 1.00
  Class 1: 183.69


In [35]:
# [Cell 14] Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

# Keep unscaled for tree-based models
X_train_trees = X_train_balanced.values if hasattr(X_train_balanced, 'values') else X_train_balanced
X_test_trees = X_test.values if hasattr(X_test, 'values') else X_test

print("Feature scaling complete.")
print(f"Scaled training shape: {X_train_scaled.shape}")
print(f"Scaled test shape: {X_test_scaled.shape}")

# Save scaler
scaler_path = OUTPUT_DIR / "feature_scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved to: {scaler_path}")


Feature scaling complete.
Scaled training shape: (778332, 29)
Scaled test shape: (194014, 29)

Scaler saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/feature_scaler.joblib


## Step 5: Model Training

Train 4 models optimized for HIGH RECALL (detecting all timestomped files).

Key configurations:
- High class weights for minority class
- Lower decision thresholds where applicable
- Ensemble methods that handle imbalance well


In [36]:
# [Cell 16] Evaluation Helper Functions

def evaluate_model(model, X_test, y_test, model_name, use_threshold=0.5):
    """
    Evaluate model with focus on Recall.
    No ROC-AUC due to extreme class imbalance.
    """
    # Get probabilities
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        # Use custom threshold for predictions (lower threshold = higher recall)
        y_pred = (y_prob >= use_threshold).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_prob = None
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        # Handle edge case where only one class present
        tn, fp, fn, tp = 0, 0, 0, 0
        if len(cm) == 1:
            if y_test.iloc[0] == 0:
                tn = cm[0, 0]
            else:
                tp = cm[0, 0]
    
    metrics = {
        'Model': model_name,
        'Threshold': use_threshold,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn
    }
    
    return metrics, y_pred, y_prob


def print_evaluation(metrics):
    """Print formatted evaluation results."""
    print(f"\n{'='*60}")
    print(f"Model: {metrics['Model']}")
    print(f"{'='*60}")
    print(f"Threshold: {metrics['Threshold']:.2f}")
    print(f"Accuracy:  {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall:    {metrics['Recall']:.4f}  <-- PRIORITY METRIC")
    print(f"F1-Score:  {metrics['F1-Score']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TP (Correctly detected): {metrics['TP']:,}")
    print(f"  FN (Missed detections):  {metrics['FN']:,}  <-- MUST BE LOW")
    print(f"  FP (False alarms):       {metrics['FP']:,}")
    print(f"  TN (Correctly normal):   {metrics['TN']:,}")


def find_optimal_threshold(model, X_test, y_test, target_recall=0.95):
    """
    Find threshold that achieves target recall while maximizing precision.
    """
    if not hasattr(model, 'predict_proba'):
        return 0.5
    
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Try different thresholds
    best_threshold = 0.5
    best_precision = 0.0
    
    for threshold in np.arange(0.01, 0.99, 0.01):
        y_pred = (y_prob >= threshold).astype(int)
        recall = recall_score(y_test, y_pred, zero_division=0)
        precision = precision_score(y_test, y_pred, zero_division=0)
        
        if recall >= target_recall and precision > best_precision:
            best_precision = precision
            best_threshold = threshold
    
    return best_threshold


print("Evaluation helper functions defined.")


Evaluation helper functions defined.


In [37]:
# [Cell 17] Train Random Forest

print("Training Random Forest...")
print("Configured for HIGH RECALL (detect all timestomped files)")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced_subsample',  # Handle imbalance per tree
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train_trees, y_train_balanced)

# Evaluate with default threshold
rf_metrics_default, rf_pred, rf_prob = evaluate_model(
    rf_model, X_test_trees, y_test, "Random Forest (threshold=0.5)", use_threshold=0.5
)
print_evaluation(rf_metrics_default)

# Find optimal threshold for 95% recall
if rf_prob is not None and y_test.sum() > 0:
    optimal_threshold = find_optimal_threshold(rf_model, X_test_trees, y_test, target_recall=0.95)
    print(f"\nOptimal threshold for 95% recall: {optimal_threshold:.2f}")
    
    rf_metrics_optimal, rf_pred_opt, _ = evaluate_model(
        rf_model, X_test_trees, y_test, f"Random Forest (threshold={optimal_threshold:.2f})", 
        use_threshold=optimal_threshold
    )
    print_evaluation(rf_metrics_optimal)
    rf_metrics = rf_metrics_optimal
else:
    rf_metrics = rf_metrics_default

# Save model
rf_path = OUTPUT_DIR / "model_random_forest.joblib"
joblib.dump(rf_model, rf_path)
print(f"\nModel saved to: {rf_path}")


Training Random Forest...
Configured for HIGH RECALL (detect all timestomped files)

Model: Random Forest (threshold=0.5)
Threshold: 0.50
Accuracy:  1.0000
Precision: 0.6667
Recall:    0.6667  <-- PRIORITY METRIC
F1-Score:  0.6667

Confusion Matrix:
  TP (Correctly detected): 4
  FN (Missed detections):  2  <-- MUST BE LOW
  FP (False alarms):       2
  TN (Correctly normal):   194,006

Optimal threshold for 95% recall: 0.15

Model: Random Forest (threshold=0.15)
Threshold: 0.15
Accuracy:  1.0000
Precision: 0.4286
Recall:    1.0000  <-- PRIORITY METRIC
F1-Score:  0.6000

Confusion Matrix:
  TP (Correctly detected): 6
  FN (Missed detections):  0  <-- MUST BE LOW
  FP (False alarms):       8
  TN (Correctly normal):   194,000

Model saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_random_forest.joblib


In [38]:
# [Cell 18] Train XGBoost

print("Training XGBoost...")
print("Configured for HIGH RECALL with scale_pos_weight")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # Handle imbalance
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',  # Area under precision-recall curve
    use_label_encoder=False
)

xgb_model.fit(X_train_trees, y_train_balanced)

# Evaluate with default threshold
xgb_metrics_default, xgb_pred, xgb_prob = evaluate_model(
    xgb_model, X_test_trees, y_test, "XGBoost (threshold=0.5)", use_threshold=0.5
)
print_evaluation(xgb_metrics_default)

# Find optimal threshold for 95% recall
if xgb_prob is not None and y_test.sum() > 0:
    optimal_threshold = find_optimal_threshold(xgb_model, X_test_trees, y_test, target_recall=0.95)
    print(f"\nOptimal threshold for 95% recall: {optimal_threshold:.2f}")
    
    xgb_metrics_optimal, xgb_pred_opt, _ = evaluate_model(
        xgb_model, X_test_trees, y_test, f"XGBoost (threshold={optimal_threshold:.2f})",
        use_threshold=optimal_threshold
    )
    print_evaluation(xgb_metrics_optimal)
    xgb_metrics = xgb_metrics_optimal
else:
    xgb_metrics = xgb_metrics_default

# Save model
xgb_path = OUTPUT_DIR / "model_xgboost.joblib"
joblib.dump(xgb_model, xgb_path)
print(f"\nModel saved to: {xgb_path}")


Training XGBoost...
Configured for HIGH RECALL with scale_pos_weight

Model: XGBoost (threshold=0.5)
Threshold: 0.50
Accuracy:  1.0000
Precision: 0.5556
Recall:    0.8333  <-- PRIORITY METRIC
F1-Score:  0.6667

Confusion Matrix:
  TP (Correctly detected): 5
  FN (Missed detections):  1  <-- MUST BE LOW
  FP (False alarms):       4
  TN (Correctly normal):   194,004

Optimal threshold for 95% recall: 0.04

Model: XGBoost (threshold=0.04)
Threshold: 0.04
Accuracy:  0.9999
Precision: 0.3750
Recall:    1.0000  <-- PRIORITY METRIC
F1-Score:  0.5455

Confusion Matrix:
  TP (Correctly detected): 6
  FN (Missed detections):  0  <-- MUST BE LOW
  FP (False alarms):       10
  TN (Correctly normal):   193,998

Model saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_xgboost.joblib


In [39]:
# [Cell 19] Train LightGBM

print("Training LightGBM...")
print("Configured for HIGH RECALL with class_weight")

lgbm_model = LGBMClassifier(
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(X_train_trees, y_train_balanced)

# Evaluate with default threshold
lgbm_metrics_default, lgbm_pred, lgbm_prob = evaluate_model(
    lgbm_model, X_test_trees, y_test, "LightGBM (threshold=0.5)", use_threshold=0.5
)
print_evaluation(lgbm_metrics_default)

# Find optimal threshold for 95% recall
if lgbm_prob is not None and y_test.sum() > 0:
    optimal_threshold = find_optimal_threshold(lgbm_model, X_test_trees, y_test, target_recall=0.95)
    print(f"\nOptimal threshold for 95% recall: {optimal_threshold:.2f}")
    
    lgbm_metrics_optimal, lgbm_pred_opt, _ = evaluate_model(
        lgbm_model, X_test_trees, y_test, f"LightGBM (threshold={optimal_threshold:.2f})",
        use_threshold=optimal_threshold
    )
    print_evaluation(lgbm_metrics_optimal)
    lgbm_metrics = lgbm_metrics_optimal
else:
    lgbm_metrics = lgbm_metrics_default

# Save model
lgbm_path = OUTPUT_DIR / "model_lightgbm.joblib"
joblib.dump(lgbm_model, lgbm_path)
print(f"\nModel saved to: {lgbm_path}")


Training LightGBM...
Configured for HIGH RECALL with class_weight

Model: LightGBM (threshold=0.5)
Threshold: 0.50
Accuracy:  1.0000
Precision: 0.6250
Recall:    0.8333  <-- PRIORITY METRIC
F1-Score:  0.7143

Confusion Matrix:
  TP (Correctly detected): 5
  FN (Missed detections):  1  <-- MUST BE LOW
  FP (False alarms):       3
  TN (Correctly normal):   194,005

Optimal threshold for 95% recall: 0.05

Model: LightGBM (threshold=0.05)
Threshold: 0.05
Accuracy:  1.0000
Precision: 0.6000
Recall:    1.0000  <-- PRIORITY METRIC
F1-Score:  0.7500

Confusion Matrix:
  TP (Correctly detected): 6
  FN (Missed detections):  0  <-- MUST BE LOW
  FP (False alarms):       4
  TN (Correctly normal):   194,004

Model saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_lightgbm.joblib


In [40]:
# [Cell 20] Train Logistic Regression

print("Training Logistic Regression...")
print("Configured for HIGH RECALL with class_weight")

lr_model = LogisticRegression(
    C=0.1,  # Lower C = more regularization
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
    solver='saga',  # Works well with imbalanced data
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train_balanced)

# Evaluate with default threshold
lr_metrics_default, lr_pred, lr_prob = evaluate_model(
    lr_model, X_test_scaled, y_test, "Logistic Regression (threshold=0.5)", use_threshold=0.5
)
print_evaluation(lr_metrics_default)

# Find optimal threshold for 95% recall
if lr_prob is not None and y_test.sum() > 0:
    optimal_threshold = find_optimal_threshold(lr_model, X_test_scaled, y_test, target_recall=0.95)
    print(f"\nOptimal threshold for 95% recall: {optimal_threshold:.2f}")
    
    lr_metrics_optimal, lr_pred_opt, _ = evaluate_model(
        lr_model, X_test_scaled, y_test, f"Logistic Regression (threshold={optimal_threshold:.2f})",
        use_threshold=optimal_threshold
    )
    print_evaluation(lr_metrics_optimal)
    lr_metrics = lr_metrics_optimal
else:
    lr_metrics = lr_metrics_default

# Save model
lr_path = OUTPUT_DIR / "model_logistic_regression.joblib"
joblib.dump(lr_model, lr_path)
print(f"\nModel saved to: {lr_path}")


Training Logistic Regression...
Configured for HIGH RECALL with class_weight

Model: Logistic Regression (threshold=0.5)
Threshold: 0.50
Accuracy:  0.9926
Precision: 0.0041
Recall:    1.0000  <-- PRIORITY METRIC
F1-Score:  0.0083

Confusion Matrix:
  TP (Correctly detected): 6
  FN (Missed detections):  0  <-- MUST BE LOW
  FP (False alarms):       1,441
  TN (Correctly normal):   192,567

Optimal threshold for 95% recall: 0.98

Model: Logistic Regression (threshold=0.98)
Threshold: 0.98
Accuracy:  0.9980
Precision: 0.0151
Recall:    1.0000  <-- PRIORITY METRIC
F1-Score:  0.0298

Confusion Matrix:
  TP (Correctly detected): 6
  FN (Missed detections):  0  <-- MUST BE LOW
  FP (False alarms):       391
  TN (Correctly normal):   193,617

Model saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_logistic_regression.joblib


## Step 6: Model Comparison

Compare all models with focus on RECALL (must detect all known timestomped files).


In [41]:
# [Cell 22] Model Comparison

all_metrics = [rf_metrics, xgb_metrics, lgbm_metrics, lr_metrics]
df_comparison = pd.DataFrame(all_metrics)

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Sort by Recall (our priority metric)
df_comparison_sorted = df_comparison.sort_values('Recall', ascending=False)

print("\nRanked by RECALL (Priority Metric):")
print("-" * 80)
display_cols = ['Model', 'Recall', 'Precision', 'F1-Score', 'Accuracy', 'TP', 'FN', 'FP']
print(df_comparison_sorted[display_cols].to_string(index=False))

print("\n\nInterpretation:")
print("-" * 80)
for _, row in df_comparison_sorted.iterrows():
    total_positive = row['TP'] + row['FN']
    if total_positive > 0:
        print(f"{row['Model']}:")
        print(f"  - Detected {row['TP']}/{total_positive} timestomped files ({row['Recall']*100:.1f}%)")
        print(f"  - Missed {row['FN']} timestomped files")
        print(f"  - {row['FP']} false positives")

# Best model by Recall
best_idx = df_comparison['Recall'].idxmax()
best_model_name = df_comparison.loc[best_idx, 'Model']
best_recall = df_comparison.loc[best_idx, 'Recall']

print(f"\n*** BEST MODEL (by Recall): {best_model_name} ***")
print(f"    Recall: {best_recall:.4f}")

# Save comparison
comparison_path = OUTPUT_DIR / "model_comparison.csv"
df_comparison.to_csv(comparison_path, index=False)
print(f"\nComparison saved to: {comparison_path}")


MODEL COMPARISON SUMMARY

Ranked by RECALL (Priority Metric):
--------------------------------------------------------------------------------
                               Model  Recall  Precision  F1-Score  Accuracy  TP  FN  FP
      Random Forest (threshold=0.15)     1.0   0.428571  0.600000  0.999959   6   0   8
            XGBoost (threshold=0.04)     1.0   0.375000  0.545455  0.999948   6   0  10
           LightGBM (threshold=0.05)     1.0   0.600000  0.750000  0.999979   6   0   4
Logistic Regression (threshold=0.98)     1.0   0.015113  0.029777  0.997985   6   0 391


Interpretation:
--------------------------------------------------------------------------------
Random Forest (threshold=0.15):
  - Detected 6/6 timestomped files (100.0%)
  - Missed 0 timestomped files
  - 8 false positives
XGBoost (threshold=0.04):
  - Detected 6/6 timestomped files (100.0%)
  - Missed 0 timestomped files
  - 10 false positives
LightGBM (threshold=0.05):
  - Detected 6/6 timestomped files (10

In [42]:
# [Cell 23] Feature Importance Analysis

print("=" * 70)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Random Forest importance
rf_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Features (Random Forest):")
print(rf_importance.head(15).to_string(index=False))

# XGBoost importance
xgb_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Features (XGBoost):")
print(xgb_importance.head(15).to_string(index=False))

# Save importance
importance_path = OUTPUT_DIR / "feature_importance.csv"
rf_importance.to_csv(importance_path, index=False)
print(f"\nFeature importance saved to: {importance_path}")


FEATURE IMPORTANCE ANALYSIS

Top 15 Features (Random Forest):
                  Feature  Importance
       num_logfile_events    0.174963
num_update_resident_value    0.170335
     num_creation_changes    0.129565
        mean_jump_seconds    0.091694
             total_events    0.076961
      num_usn_file_create    0.058940
max_backward_jump_seconds    0.057996
       num_backward_jumps    0.052976
      has_usn_file_create    0.041365
    max_inter_event_delta    0.031959
  event_time_span_seconds    0.023889
            num_usn_close    0.023547
    num_timestamp_changes    0.018260
       num_usnjrnl_events    0.014774
   mean_inter_event_delta    0.014392

Top 15 Features (XGBoost):
                   Feature  Importance
        num_logfile_events    0.794078
      num_creation_changes    0.102603
         mean_jump_seconds    0.033912
       has_usn_file_create    0.021118
 num_update_resident_value    0.014000
             num_usn_close    0.011250
num_zero_nanosecond_events   

## Step 7: Validation on Held-Out Datasets

Test trained models on the 5 validation datasets:
1. 02-APT19
2. 09-APT40
3. 12-Kimsuky (3 known timestomped files)
4. 13-Winnti731
5. LoneWolf (12 known timestomped files)


In [43]:
# [Cell 25] Validation Function

def validate_on_dataset(dataset_name, models_dict, feature_columns, scaler, df_gt, use_threshold=0.3):
    """
    Validate all models on a single validation dataset using ground truth labels.
    
    Parameters:
        dataset_name: Name of the validation dataset
        models_dict: Dictionary of trained models
        feature_columns: List of feature column names
        scaler: Fitted StandardScaler for Logistic Regression
        df_gt: Ground truth DataFrame
        use_threshold: Decision threshold (lower = more detections)
    
    Returns:
        DataFrame with validation results, and detailed predictions
    """
    print(f"\n{'='*70}")
    print(f"VALIDATING ON: {dataset_name}")
    print(f"{'='*70}")
    
    # Load validation features
    features_path = PHASE3_DIR / f"file_features_{dataset_name}.csv"
    
    if not features_path.exists():
        print(f"  ERROR: Features file not found")
        return None, None
    
    df_val = pd.read_csv(features_path, low_memory=False)
    
    # Match ground truth labels
    df_val_labeled, matched = match_ground_truth(df_val, df_gt, dataset_name)
    
    expected = len(df_gt[df_gt['dataID'] == dataset_name])
    
    print(f"  Total files: {len(df_val_labeled):,}")
    print(f"  Known timestomped (found): {matched}")
    print(f"  Known timestomped (expected): {expected}")
    
    # Prepare features
    X_val = df_val_labeled[feature_columns].copy()
    X_val = X_val.fillna(0)
    bool_cols = X_val.select_dtypes(include=['bool']).columns
    X_val[bool_cols] = X_val[bool_cols].astype(int)
    
    # Labels
    y_val = df_val_labeled['is_timestomped'].astype(int)
    
    print(f"  Class 0 (Normal): {(y_val == 0).sum():,}")
    print(f"  Class 1 (Timestomped): {(y_val == 1).sum()}")
    
    # Prepare scaled features
    X_val_scaled = scaler.transform(X_val)
    X_val_trees = X_val.values
    
    # Evaluate each model
    results = []
    all_predictions = {}
    
    for model_name, model in models_dict.items():
        # Use appropriate features
        if "Logistic" in model_name:
            X_eval = X_val_scaled
        else:
            X_eval = X_val_trees
        
        # Get predictions
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_eval)[:, 1]
            y_pred = (y_prob >= use_threshold).astype(int)
        else:
            y_pred = model.predict(X_eval)
            y_prob = None
        
        # Confusion matrix
        if y_val.sum() > 0:  # Only if there are positive samples
            cm = confusion_matrix(y_val, y_pred)
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
            else:
                tn, fp, fn, tp = cm[0,0], 0, 0, 0
        else:
            # No ground truth for this dataset
            tp, fn = 0, 0
            tn = (y_pred == 0).sum()
            fp = (y_pred == 1).sum()
        
        result = {
            'Dataset': dataset_name,
            'Model': model_name,
            'Total_Files': len(y_val),
            'GT_Positive': (y_val == 1).sum(),
            'GT_Negative': (y_val == 0).sum(),
            'Accuracy': accuracy_score(y_val, y_pred) if y_val.sum() > 0 else np.nan,
            'Precision': precision_score(y_val, y_pred, zero_division=0),
            'Recall': recall_score(y_val, y_pred, zero_division=0),
            'F1-Score': f1_score(y_val, y_pred, zero_division=0),
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn
        }
        results.append(result)
        
        # Store predictions
        all_predictions[model_name] = {
            'y_pred': y_pred,
            'y_prob': y_prob
        }
        
        print(f"\n  {model_name}:")
        if y_val.sum() > 0:
            print(f"    Recall: {result['Recall']:.4f} ({tp}/{tp+fn} detected)")
            print(f"    Precision: {result['Precision']:.4f}")
        print(f"    TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
    
    return pd.DataFrame(results), df_val_labeled


print("Validation function defined.")


Validation function defined.


In [44]:
# [Cell 26] Prepare Models Dictionary

models_dict = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "Logistic Regression": lr_model
}

print("Models ready for validation:")
for name in models_dict.keys():
    print(f"  - {name}")


Models ready for validation:
  - Random Forest
  - XGBoost
  - LightGBM
  - Logistic Regression


In [45]:
# [Cell 27] Validate on 02-APT19
results_02APT19, df_02APT19 = validate_on_dataset(
    "02-APT19", models_dict, feature_columns, scaler, df_ground_truth, use_threshold=0.3
)



VALIDATING ON: 02-APT19
  Total files: 21,336
  Known timestomped (found): 0
  Known timestomped (expected): 0
  Class 0 (Normal): 21,336
  Class 1 (Timestomped): 0

  Random Forest:
    TP: 0, FP: 5, TN: 21331, FN: 0

  XGBoost:
    TP: 0, FP: 5, TN: 21331, FN: 0

  LightGBM:
    TP: 0, FP: 4, TN: 21332, FN: 0

  Logistic Regression:
    TP: 0, FP: 849, TN: 20487, FN: 0


In [46]:
# [Cell 28] Validate on 09-APT40
results_09APT40, df_09APT40 = validate_on_dataset(
    "09-APT40", models_dict, feature_columns, scaler, df_ground_truth, use_threshold=0.3
)



VALIDATING ON: 09-APT40
  Total files: 21,672
  Known timestomped (found): 0
  Known timestomped (expected): 0
  Class 0 (Normal): 21,672
  Class 1 (Timestomped): 0

  Random Forest:
    TP: 0, FP: 0, TN: 21672, FN: 0

  XGBoost:
    TP: 0, FP: 1, TN: 21671, FN: 0

  LightGBM:
    TP: 0, FP: 1, TN: 21671, FN: 0

  Logistic Regression:
    TP: 0, FP: 1091, TN: 20581, FN: 0


In [47]:
# [Cell 29] Validate on 12-Kimsuky
results_12Kimsuky, df_12Kimsuky = validate_on_dataset(
    "12-Kimsuky", models_dict, feature_columns, scaler, df_ground_truth, use_threshold=0.3
)



VALIDATING ON: 12-Kimsuky
  Total files: 14,869
  Known timestomped (found): 3
  Known timestomped (expected): 3
  Class 0 (Normal): 14,866
  Class 1 (Timestomped): 3

  Random Forest:
    Recall: 0.6667 (2/3 detected)
    Precision: 0.6667
    TP: 2, FP: 1, TN: 14865, FN: 1

  XGBoost:
    Recall: 0.6667 (2/3 detected)
    Precision: 0.6667
    TP: 2, FP: 1, TN: 14865, FN: 1

  LightGBM:
    Recall: 0.6667 (2/3 detected)
    Precision: 1.0000
    TP: 2, FP: 0, TN: 14866, FN: 1

  Logistic Regression:
    Recall: 1.0000 (3/3 detected)
    Precision: 0.0038
    TP: 3, FP: 785, TN: 14081, FN: 0


In [48]:
# [Cell 30] Validate on 13-Winnti731
results_13Winnti731, df_13Winnti731 = validate_on_dataset(
    "13-Winnti731", models_dict, feature_columns, scaler, df_ground_truth, use_threshold=0.3
)



VALIDATING ON: 13-Winnti731
  Total files: 16,741
  Known timestomped (found): 0
  Known timestomped (expected): 0
  Class 0 (Normal): 16,741
  Class 1 (Timestomped): 0

  Random Forest:
    TP: 0, FP: 2, TN: 16739, FN: 0

  XGBoost:
    TP: 0, FP: 2, TN: 16739, FN: 0

  LightGBM:
    TP: 0, FP: 2, TN: 16739, FN: 0

  Logistic Regression:
    TP: 0, FP: 780, TN: 15961, FN: 0


In [49]:
# [Cell 31] Validate on LoneWolf
results_LoneWolf, df_LoneWolf = validate_on_dataset(
    "LoneWolf", models_dict, feature_columns, scaler, df_ground_truth, use_threshold=0.3
)




VALIDATING ON: LoneWolf
  Total files: 17,897
  Known timestomped (found): 12
  Known timestomped (expected): 12
  Class 0 (Normal): 17,885
  Class 1 (Timestomped): 12

  Random Forest:
    Recall: 0.0000 (0/12 detected)
    Precision: 0.0000
    TP: 0, FP: 6, TN: 17879, FN: 12

  XGBoost:
    Recall: 0.0000 (0/12 detected)
    Precision: 0.0000
    TP: 0, FP: 3, TN: 17882, FN: 12

  LightGBM:
    Recall: 0.0000 (0/12 detected)
    Precision: 0.0000
    TP: 0, FP: 1, TN: 17884, FN: 12

  Logistic Regression:
    Recall: 1.0000 (12/12 detected)
    Precision: 0.0262
    TP: 12, FP: 446, TN: 17439, FN: 0


## Step 8: Validation Summary

Aggregate and analyze validation results across all held-out datasets.
Focus on detecting all known timestomped files (Recall).


In [50]:
# [Cell 33] Aggregate Validation Results

print("=" * 80)
print("VALIDATION RESULTS SUMMARY")
print("=" * 80)

# Collect all validation results
all_val_results = []
for result in [results_02APT19, results_09APT40, results_12Kimsuky, results_13Winnti731, results_LoneWolf]:
    if result is not None:
        all_val_results.append(result)

if all_val_results:
    df_val_summary = pd.concat(all_val_results, ignore_index=True)
    
    # Only consider datasets with ground truth
    df_val_with_gt = df_val_summary[df_val_summary['GT_Positive'] > 0]
    
    if len(df_val_with_gt) > 0:
        print("\nDatasets WITH ground truth labels:")
        print("-" * 70)
        
        # Summary per model (only for datasets with ground truth)
        model_summary = df_val_with_gt.groupby('Model').agg({
            'TP': 'sum',
            'FP': 'sum',
            'TN': 'sum',
            'FN': 'sum',
            'GT_Positive': 'sum'
        }).reset_index()
        
        # Calculate aggregate metrics
        model_summary['Recall'] = model_summary['TP'] / (model_summary['TP'] + model_summary['FN'])
        model_summary['Precision'] = model_summary['TP'] / (model_summary['TP'] + model_summary['FP'])
        model_summary['Precision'] = model_summary['Precision'].fillna(0)
        
        model_summary = model_summary.sort_values('Recall', ascending=False)
        
        print("\nAggregate Performance (across datasets with ground truth):")
        print(model_summary[['Model', 'Recall', 'Precision', 'TP', 'FN', 'FP', 'GT_Positive']].to_string(index=False))
        
        print("\n\nPer-Dataset Recall:")
        print("-" * 70)
        pivot_recall = df_val_with_gt.pivot(index='Dataset', columns='Model', values='Recall')
        print(pivot_recall.to_string())
    
    # Show datasets without ground truth
    df_val_no_gt = df_val_summary[df_val_summary['GT_Positive'] == 0]
    if len(df_val_no_gt) > 0:
        print("\n\nDatasets WITHOUT ground truth (showing predictions only):")
        print("-" * 70)
        no_gt_summary = df_val_no_gt.groupby('Model')['FP'].sum().reset_index()
        no_gt_summary.columns = ['Model', 'Predicted_Positive']
        print(no_gt_summary.to_string(index=False))
        print("(These are files predicted as timestomped - cannot verify without ground truth)")
    
    # Save validation results
    val_path = OUTPUT_DIR / "validation_results.csv"
    df_val_summary.to_csv(val_path, index=False)
    print(f"\n\nValidation results saved to: {val_path}")
else:
    print("No validation results available.")


VALIDATION RESULTS SUMMARY

Datasets WITH ground truth labels:
----------------------------------------------------------------------

Aggregate Performance (across datasets with ground truth):
              Model   Recall  Precision  TP  FN   FP  GT_Positive
Logistic Regression 1.000000   0.012039  15   0 1231           15
           LightGBM 0.133333   0.666667   2  13    1           15
      Random Forest 0.133333   0.222222   2  13    7           15
            XGBoost 0.133333   0.333333   2  13    4           15


Per-Dataset Recall:
----------------------------------------------------------------------
Model       LightGBM  Logistic Regression  Random Forest   XGBoost
Dataset                                                           
12-Kimsuky  0.666667                  1.0       0.666667  0.666667
LoneWolf    0.000000                  1.0       0.000000  0.000000


Datasets WITHOUT ground truth (showing predictions only):
-------------------------------------------------------

In [51]:
# [Cell 34] Detailed Analysis: Which Files Were Detected/Missed

print("=" * 80)
print("DETAILED DETECTION ANALYSIS")
print("=" * 80)

# Focus on datasets with ground truth
validation_data = {
    '12-Kimsuky': df_12Kimsuky,
    'LoneWolf': df_LoneWolf
}

for dataset_name, df_data in validation_data.items():
    if df_data is None:
        continue
    
    gt_files = df_data[df_data['is_timestomped'] == 1]
    
    if len(gt_files) == 0:
        continue
    
    print(f"\n{dataset_name}: {len(gt_files)} known timestomped files")
    print("-" * 60)
    
    # Get predictions from best model (by recall on test set)
    best_model = xgb_model  # Or whichever had best recall
    
    X_check = df_data[feature_columns].fillna(0)
    bool_cols = X_check.select_dtypes(include=['bool']).columns
    X_check[bool_cols] = X_check[bool_cols].astype(int)
    
    y_prob = best_model.predict_proba(X_check.values)[:, 1]
    df_data['pred_prob'] = y_prob
    df_data['pred_detected'] = (y_prob >= 0.3).astype(int)
    
    # Show ground truth files and their detection status
    gt_files_with_pred = df_data[df_data['is_timestomped'] == 1][
        ['FileName', 'FilePath', 'pred_prob', 'pred_detected']
    ]
    
    print("\nKnown timestomped files:")
    for _, row in gt_files_with_pred.iterrows():
        status = "DETECTED" if row['pred_detected'] == 1 else "MISSED"
        print(f"  [{status}] {row['FileName']} (prob: {row['pred_prob']:.3f})")
    
    detected = gt_files_with_pred['pred_detected'].sum()
    total = len(gt_files_with_pred)
    print(f"\nSummary: {detected}/{total} detected ({detected/total*100:.1f}% recall)")


DETAILED DETECTION ANALYSIS

12-Kimsuky: 3 known timestomped files
------------------------------------------------------------

Known timestomped files:
  [DETECTED] svcsmon_ko.dll (prob: 1.000)
  [DETECTED] svcsmon.exe (prob: 1.000)
  [MISSED] wsmss.exe (prob: 0.000)

Summary: 2/3 detected (66.7% recall)

LoneWolf: 12 known timestomped files
------------------------------------------------------------

Known timestomped files:
  [MISSED] Sheep.jpg (prob: 0.000)
  [MISSED] RedGuns.jpg (prob: 0.000)
  [MISSED] DarkWolf.png (prob: 0.000)
  [MISSED] DemLogic.jpg (prob: 0.000)
  [MISSED] DeathToll.jpg (prob: 0.000)
  [MISSED] Planning.docx (prob: 0.000)
  [MISSED] CubaDearmed.jpg (prob: 0.000)
  [MISSED] Huckleberry.png (prob: 0.000)
  [MISSED] MyTiredHead.jpg (prob: 0.000)
  [MISSED] BladeofGrass.jpg (prob: 0.000)
  [MISSED] HoldMyTidePod.jpg (prob: 0.000)
  [MISSED] AIRPORT INFORMATION.docx (prob: 0.000)

Summary: 0/12 detected (0.0% recall)


In [52]:
# [Cell 35] Best Model Selection

print("=" * 80)
print("BEST MODEL SELECTION")
print("=" * 80)

# Test set performance
print("\nTest Set Performance (Ranked by Recall):")
test_perf = df_comparison[['Model', 'Recall', 'Precision', 'F1-Score', 'TP', 'FN']].copy()
test_perf = test_perf.sort_values('Recall', ascending=False)
print(test_perf.to_string(index=False))

# Validation performance (if available)
if all_val_results and len(df_val_with_gt) > 0:
    print("\nValidation Set Performance (Aggregate, Ranked by Recall):")
    val_perf = model_summary[['Model', 'Recall', 'Precision', 'TP', 'FN']].copy()
    val_perf = val_perf.sort_values('Recall', ascending=False)
    print(val_perf.to_string(index=False))
    
    # Combined selection
    best_val_model = val_perf.iloc[0]['Model']
    best_val_recall = val_perf.iloc[0]['Recall']
    
    print(f"\n*** RECOMMENDED MODEL: {best_val_model} ***")
    print(f"    Validation Recall: {best_val_recall:.4f}")
    print(f"    (Detected {int(val_perf.iloc[0]['TP'])} out of {int(val_perf.iloc[0]['TP'] + val_perf.iloc[0]['FN'])} known timestomped files)")


BEST MODEL SELECTION

Test Set Performance (Ranked by Recall):
                               Model  Recall  Precision  F1-Score  TP  FN
      Random Forest (threshold=0.15)     1.0   0.428571  0.600000   6   0
            XGBoost (threshold=0.04)     1.0   0.375000  0.545455   6   0
           LightGBM (threshold=0.05)     1.0   0.600000  0.750000   6   0
Logistic Regression (threshold=0.98)     1.0   0.015113  0.029777   6   0

Validation Set Performance (Aggregate, Ranked by Recall):
              Model   Recall  Precision  TP  FN
Logistic Regression 1.000000   0.012039  15   0
           LightGBM 0.133333   0.666667   2  13
      Random Forest 0.133333   0.222222   2  13
            XGBoost 0.133333   0.333333   2  13

*** RECOMMENDED MODEL: Logistic Regression ***
    Validation Recall: 1.0000
    (Detected 15 out of 15 known timestomped files)


In [53]:
# [Cell 36] Save Final Artifacts

print("=" * 80)
print("SAVING FINAL ARTIFACTS")
print("=" * 80)

# Save all models
print("\nSaved Models:")
for name, model in models_dict.items():
    model_filename = f"model_{name.lower().replace(' ', '_')}.joblib"
    model_path = OUTPUT_DIR / model_filename
    joblib.dump(model, model_path)
    print(f"  {name}: {model_path}")

# Save training configuration
training_config = {
    'feature_columns': feature_columns,
    'training_samples_original': len(X_train),
    'training_samples_balanced': len(X_train_balanced),
    'test_samples': len(X_test),
    'ground_truth_file': str(GROUND_TRUTH_PATH),
    'total_known_timestomped': int(df_ground_truth['is_timestomped'].sum()),
    'training_datasets': TRAINING_DATASETS,
    'validation_datasets': VALIDATION_DATASETS,
    'imbalance_ratio': float(imbalance_ratio) if imbalance_ratio != float('inf') else 'inf',
    'class_weights': {str(k): float(v) for k, v in class_weight_dict.items()},
    'recommended_threshold': 0.3
}

config_path = OUTPUT_DIR / "training_config.json"
with open(config_path, 'w') as f:
    json.dump(training_config, f, indent=2)
print(f"\nTraining config saved to: {config_path}")

print(f"\nScaler saved to: {scaler_path}")

print("\n" + "=" * 80)
print("PHASE 4 COMPLETE")
print("=" * 80)


SAVING FINAL ARTIFACTS

Saved Models:
  Random Forest: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_random_forest.joblib
  XGBoost: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_xgboost.joblib
  LightGBM: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_lightgbm.joblib
  Logistic Regression: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/model_logistic_regression.joblib

Training config saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/training_config.json

Scaler saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/feature_scaler.joblib

PHASE 4 COMPLETE


## Summary

### Key Differences from Previous Version
- Used ACTUAL ground truth labels (44 known timestomped files from LogTracker)
- No longer using computed `flag_potential_timestomp` (which caused data leakage)
- Prioritized RECALL over Precision (must detect all known timestomped files)
- Removed ROC-AUC metric (misleading with extreme class imbalance)
- Applied appropriate class balancing techniques for ~250,000:1 imbalance

### Handling Extreme Class Imbalance
- SMOTE with controlled ratio (not 1:1 to avoid too many synthetic samples)
- Class weights (sqrt of imbalance ratio)
- Lower decision thresholds (0.3 instead of 0.5)
- Stratified sampling for train-test split

### Artifacts Saved
- `model_random_forest.joblib`
- `model_xgboost.joblib`
- `model_lightgbm.joblib`
- `model_logistic_regression.joblib`
- `feature_scaler.joblib`
- `training_config.json`
- `model_comparison.csv`
- `validation_results.csv`
- `feature_importance.csv`

### Next Steps
1. Tune decision threshold for optimal Recall/Precision trade-off
2. Hyperparameter tuning (Phase 5)
3. Deploy selected model for Autopsy integration (Phase 6)


In [54]:
# [Cell 38] Final Summary Statistics

print("=" * 80)
print("PHASE 4 FINAL SUMMARY")
print("=" * 80)

print(f"\n{'Metric':<50} {'Value':>20}")
print("-" * 75)
print(f"{'Ground truth timestomped files':<50} {int(df_ground_truth['is_timestomped'].sum()):>20}")
print(f"{'Training samples (original)':<50} {len(X_train):>20,}")
print(f"{'Training samples (after SMOTE)':<50} {len(X_train_balanced):>20,}")
print(f"{'Test samples':<50} {len(X_test):>20,}")
print(f"{'Features used':<50} {len(feature_columns):>20}")
print(f"{'Models trained':<50} {len(models_dict):>20}")
print(f"{'Validation datasets':<50} {len(VALIDATION_DATASETS):>20}")
print(f"{'Class imbalance ratio':<50} {imbalance_ratio:>20,.0f}:1")
print("-" * 75)

print("\nTest Set Performance (Sorted by Recall):")
for _, row in df_comparison.sort_values('Recall', ascending=False).iterrows():
    print(f"  {row['Model']:<30} Recall: {row['Recall']:.4f}, Precision: {row['Precision']:.4f}, F1: {row['F1-Score']:.4f}")

print("\n" + "=" * 80)


PHASE 4 FINAL SUMMARY

Metric                                                            Value
---------------------------------------------------------------------------
Ground truth timestomped files                                       44
Training samples (original)                                     776,055
Training samples (after SMOTE)                                  778,332
Test samples                                                    194,014
Features used                                                        29
Models trained                                                        4
Validation datasets                                                   5
Class imbalance ratio                                            33,450:1
---------------------------------------------------------------------------

Test Set Performance (Sorted by Recall):
  Random Forest (threshold=0.15) Recall: 1.0000, Precision: 0.4286, F1: 0.6000
  XGBoost (threshold=0.04)       Recall: 1.0000, Preci